# Prodigy_MultiParam_Study — R2.4 (90 esperimenti)

**Obiettivo**: capire come usare Prodigy nelle SNN, dove i requisiti differiscono dalle diffusion (LR effettivo deve essere ~2e-3 max, non 0.1+ come Prodigy default fa explodere).

**Scoperta R2.3 (motivazione)**: tutti i 5 esperimenti R2.2 sono andati in gradient explosion (961/1000 batch con grad=inf in P-A, similar negli altri). lr_eff schizzava a 0.017-0.195 → ESPLOSIONE. Solo P-E con cosine modulava lr_eff in calo. Vedi `PRODIGY_DEEP_STUDY.md` parte 3 §17.

## Setup

- 3 LR ∈ {1.0, 0.5, 0.1} × 10 varianti Prodigy × 3 scenari = **90 run × 10 ep × 100 step**
- Base canonical sempre: `betas=(0.9, 0.99)` (W1) + `use_bias_correction=True` (W3) + `safeguard_warmup=True` + `weight_decay=0.01`
- BASELINE_BPTT_864p_PRE_EVENTPROP arch (864p, lambda_sr=0.5)
- Push per-run dopo ogni esperimento (pattern T30 BEST notebook)
- Results: `results/Prodigy_Study/MultiParam/<scenario>/<tag>/`

## 10 varianti (per LR fisso)

| ID | d_coef | d0 | growth_rate | scheduler | Logica |
|---|---:|---:|---:|---|---|
| V01 | 1.0 | 1e-6 | inf | none | Reference canonical |
| V02 | 0.5 | 1e-6 | inf | none | Brake leggero |
| V03 | 0.1 | 1e-6 | inf | none | Brake medio |
| V04 | 0.01 | 1e-6 | inf | none | Brake forte |
| V05 | 1.0 | 1e-6 | 1.02 | none | Growth cap smooth (paper) |
| V06 | 1.0 | 1e-6 | 1.01 | none | Growth cap stretto |
| V07 | 1.0 | 1e-4 | inf | none | d0 preheat (canonical kohya) |
| V08 | 1.0 | 1e-6 | inf | cosine_no_restart | Cosine decay |
| V09 | 0.1 | 1e-6 | 1.02 | cosine_no_restart | Combo brake+cap+cosine |
| V10 | 2.0 | 1e-6 | inf | none | d_coef=2 canonical kohya |

## 3 scenari training

1. **highway**: solo highway (cache F2 esistente, baseline storica)
2. **mixed**: 4 scenari (highway 40% + urban 30% + truck 20% + mixed 10%), no cut-in
3. **full**: SCENARIO_MIX default config + cut_in_ratio=0.2 (come primissimi tentativi)

## Tag naming

`R24_<scenario>_lr<L>_<V>` es. `R24_highway_lr1.0_V01`, `R24_full_lr0.1_V09`.

**Stima tempo Azure**: ~15h totali (90 × 10 min). Push per-run + SKIP_IF_EXISTS robusto.

**Reference**: `PRODIGY_DEEP_STUDY.md` parte 1+2+3.


In [ ]:
# Cell 1 -- Bootstrap + ENV check
import sys, os, subprocess
import importlib.util as _imu

# Install deps (idempotent)
for pkg in ['prodigyopt', 'pandas', 'matplotlib']:
    if _imu.find_spec(pkg) is None:
        print(f'  installing {pkg}...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)

# Verifica file critici
for f in ['train.py', 'core/network.py', 'config.py', 'data/generator.py']:
    assert os.path.isfile(f), f'MISSING: {f}'
    print(f'  [OK] {f}')

# Prodigy API
from prodigyopt import Prodigy
import inspect
sig = inspect.signature(Prodigy.__init__)
for p in ['safeguard_warmup','growth_rate','d_coef','d0','betas','use_bias_correction','weight_decay']:
    assert p in sig.parameters, f'MISSING Prodigy param: {p}'
print('[OK] Prodigy API: 7 param richiesti presenti')

# train.py CLI flags
help_txt = subprocess.run([sys.executable, 'train.py', '--help'],
                           capture_output=True, text=True, encoding='utf-8',
                           errors='replace').stdout
for flag in ['--prodigy_safeguard_warmup','--prodigy_growth_rate','--prodigy_d_coef',
             '--prodigy_betas','--prodigy_use_bias_correction','--prodigy_d0',
             '--prodigy_weight_decay']:
    assert flag in help_txt, f'MISSING train.py flag: {flag}'
    print(f'  [OK] {flag}')
assert 'cosine_no_restart' in help_txt, 'MISSING scheduler cosine_no_restart'
print('  [OK] scheduler cosine_no_restart')

# git config
import subprocess as sp
br = sp.run(['git','branch','--show-current'], capture_output=True, text=True).stdout.strip()
print(f'\n[GIT] branch corrente: {br}')
assert br == 'Prodigy_Deep_Study', f'Wrong branch: {br} (expected Prodigy_Deep_Study)'
print('[OK] su branch Prodigy_Deep_Study, push per-run abilitato')
print('\nENV check passed.')


In [ ]:
# Cell 2 -- Generate 90 EXPERIMENTS list (3 scenari x 3 LR x 10 varianti)
import itertools

VARIANTS = [
    # ID, d_coef, d0, growth_rate, scheduler, desc
    ('V01', 1.0,   1e-6, 'inf',  'none',              'Reference canonical (W1+W3 only)'),
    ('V02', 0.5,   1e-6, 'inf',  'none',              'Brake leggero (d_coef=0.5)'),
    ('V03', 0.1,   1e-6, 'inf',  'none',              'Brake medio (d_coef=0.1)'),
    ('V04', 0.01,  1e-6, 'inf',  'none',              'Brake forte (d_coef=0.01)'),
    ('V05', 1.0,   1e-6, '1.02', 'none',              'Growth_rate cap smooth (paper 1.02)'),
    ('V06', 1.0,   1e-6, '1.01', 'none',              'Growth_rate cap stretto (1.01)'),
    ('V07', 1.0,   1e-4, 'inf',  'none',              'd0 preheat alto (canonical kohya)'),
    ('V08', 1.0,   1e-6, 'inf',  'cosine_no_restart', 'Cosine sched (decay finale)'),
    ('V09', 0.1,   1e-6, '1.02', 'cosine_no_restart', 'Combo brake + cap + cosine'),
    ('V10', 2.0,   1e-6, 'inf',  'none',              'd_coef=2 canonical kohya'),
]
LRS = [1.0, 0.5, 0.1]
SCENARIOS = [
    ('highway', 'highway',                                        0.0, 'data/cache_1500_highway_cut0.0_ou0.0.pt'),
    ('mixed',   'highway:0.4,urban:0.3,truck:0.2,mixed:0.1',        0.0, 'data/cache_1500_mixed_cut0.0_ou0.0.pt'),
    ('full',    'default',                                         0.2, 'data/cache_1500_full_cut0.2_ou0.0.pt'),
]

EXPERIMENTS = []
for (scn_name, scn_mix, cut_in, cache_path), lr, (vid, dcoef, d0, growth, sched, desc) in \
        itertools.product(SCENARIOS, LRS, VARIANTS):
    tag = f'R24_{scn_name}_lr{lr}_{vid}'
    EXPERIMENTS.append({
        'tag': tag, 'scenario': scn_name, 'scenario_mix': scn_mix, 'cut_in_ratio': cut_in,
        'cache_path': cache_path, 'lr': lr,
        'variant_id': vid, 'd_coef': dcoef, 'd0': d0,
        'growth_rate': growth, 'scheduler': sched, 'desc': desc,
    })

print(f'Total experiments: {len(EXPERIMENTS)} (atteso 90 = 3 scn x 3 lr x 10 var)')
assert len(EXPERIMENTS) == 90, f'Expected 90, got {len(EXPERIMENTS)}'
for s in ['highway','mixed','full']:
    count = sum(1 for e in EXPERIMENTS if e['scenario']==s)
    print(f'  {s}: {count} run')
print(f'\nFirst exp: {EXPERIMENTS[0]["tag"]}')
print(f'Last exp:  {EXPERIMENTS[-1]["tag"]}')
print(f'\nVARIANTS list:')
for v in VARIANTS:
    print(f'  {v[0]}: d_coef={v[1]} d0={v[2]} growth={v[3]} sched={v[4]:>20s} -- {v[5]}')


In [ ]:
# Cell 3 -- Cache check + AUTOGEN per scenari mancanti
import os, time
import torch

for scn_name, scn_mix, cut_in, cache_path in SCENARIOS:
    if os.path.isfile(cache_path):
        sz = os.path.getsize(cache_path) / 1024 / 1024
        print(f'[OK] {scn_name}: {cache_path} ({sz:.1f} MB)')
        continue
    print(f'\n[AUTOGEN] {scn_name}: {cache_path} mancante, genero...')
    print(f'  scenario_mix={scn_mix} cut_in_ratio={cut_in}')
    from data.generator import generate_dataset, parse_scenario_mix
    from config import SEED
    mix_dict = parse_scenario_mix(scn_mix)
    t0 = time.time()
    train_d = generate_dataset(1500, base_seed=SEED, scenario_mix=mix_dict,
                                cut_in_ratio=cut_in, noise_scale=0.0)
    val_d   = generate_dataset(300,  base_seed=SEED+1, scenario_mix=mix_dict,
                                cut_in_ratio=cut_in, noise_scale=0.0)
    torch.save({'train': train_d, 'val': val_d, 'seed': SEED}, cache_path)
    sz = os.path.getsize(cache_path) / 1024 / 1024
    print(f'  [SAVED] {cache_path} ({sz:.1f} MB) in {time.time()-t0:.0f}s')
print('\nAll caches ready.')


In [ ]:
# Cell 4 -- RUN sweep 90 esperimenti con PUSH per-run (pattern T30 BEST)
import subprocess, sys, time, os, shutil, glob, datetime
import pandas as pd

SKIP_IF_EXISTS = True   # RESUME safe
RESULTS_DIR = 'results/Prodigy_Study/MultiParam'
os.makedirs(RESULTS_DIR, exist_ok=True)

# Sanity: nome del messaggio commit temp
_TMP_MSG = '/tmp/r24_msg.txt' if os.path.isdir('/tmp') else 'r24_msg.txt'

def _build_cli(e):
    return [sys.executable, 'train.py',
        '--training_method', 'baseline',
        '--epochs', '10', '--max_steps_per_epoch', '100',
        '--batch_size', '8', '--val_batch_size', '32', '--seq_len', '50',
        '--cf_hidden_size', '32', '--cf_rank', '8',
        '--lambda_data', '1.0', '--lambda_phys', '0.1', '--lambda_ou', '0.05',
        '--lambda_bc', '1.0', '--lambda_sr', '0.5',
        '--scenario_mix', e['scenario_mix'],
        '--cut_in_ratio', str(e['cut_in_ratio']),
        '--noise_scale', '0.0', '--po2_enabled', '1',
        '--n_train', '1500', '--n_val', '300',
        '--max_inf_streak', '99999', '--early_stop_patience', '0',
        '--optimizer', 'prodigy',
        '--data_cache', e['cache_path'],
        '--lr', str(e['lr']), '--max_lr', str(e['lr']),
        '--scheduler', e['scheduler'],
        '--prodigy_betas', '0.9,0.99',
        '--prodigy_d_coef', str(e['d_coef']),
        '--prodigy_d0', str(e['d0']),
        '--prodigy_weight_decay', '0.01',
        '--prodigy_use_bias_correction', '1',
        '--prodigy_safeguard_warmup', '1',
        '--prodigy_growth_rate', str(e['growth_rate']),
        '--tag', e['tag']]

def _push_run(e):
    """Move checkpoints/<tag>/ -> results/Prodigy_Study/MultiParam/<scenario>/<tag>/ + git push."""
    tag = e['tag']
    src = f'checkpoints/{tag}'
    dst_parent = f'{RESULTS_DIR}/{e["scenario"]}'
    dst = f'{dst_parent}/{tag}'
    if not os.path.isdir(src):
        print(f'   [WARN push] {src} mancante, skip')
        return False
    os.makedirs(dst_parent, exist_ok=True)
    if os.path.isdir(dst): shutil.rmtree(dst)
    os.makedirs(f'{dst}/plots', exist_ok=True)
    # Copia csv + json (NO .pt per ridurre push size)
    for f in glob.glob(f'{src}/*.csv') + glob.glob(f'{src}/*.json'):
        shutil.copy2(f, dst)
    for f in glob.glob(f'{src}/plots/*.png'):
        shutil.copy2(f, f'{dst}/plots/')

    # Build commit message
    val_str = ''
    if os.path.isfile(f'{dst}/training_log.csv'):
        try:
            edf = pd.read_csv(f'{dst}/training_log.csv')
            if len(edf) > 0:
                bi = int(edf.val_total.idxmin())
                val_str = f'best val_total={edf.val_total.min():.4f} val_data={edf.val_data.iloc[bi]:.4f} (E{bi+1}/{len(edf)})'
        except Exception as ex:
            val_str = f'(log read failed: {ex})'

    ts = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
    msg = (
        f'results (R2.4 Prodigy MultiParam): {tag} ({ts})\n\n'
        f'{val_str}\n'
        f'scenario={e["scenario"]} lr={e["lr"]} d_coef={e["d_coef"]} d0={e["d0"]} '
        f'growth={e["growth_rate"]} sched={e["scheduler"]}\n'
        f'Branch Prodigy_Deep_Study\n'
    )
    with open(_TMP_MSG, 'w', encoding='utf-8') as fp:
        fp.write(msg)
    # Git operations (subprocess invece di ! per error catching robusto)
    try:
        subprocess.run(['git','add',dst], check=True, capture_output=True)
        r = subprocess.run(['git','commit','-F',_TMP_MSG], capture_output=True, text=True)
        if r.returncode != 0:
            if 'nothing to commit' in r.stdout or 'nothing to commit' in r.stderr:
                print(f'   [push] nothing to commit (already in tree?)')
                return True
            print(f'   [push] commit failed rc={r.returncode}: {r.stderr[-300:]}')
            return False
        subprocess.run(['git','pull','--no-rebase','--no-edit','origin','Prodigy_Deep_Study'],
                       capture_output=True, text=True)  # best-effort
        r2 = subprocess.run(['git','push','origin','Prodigy_Deep_Study'], capture_output=True, text=True)
        if r2.returncode != 0:
            print(f'   [push] push failed rc={r2.returncode}: {r2.stderr[-300:]}')
            return False
        print(f'   [push] OK')
        return True
    finally:
        try: os.remove(_TMP_MSG)
        except: pass

# Esecuzione
run_results = []
t_start = time.time()
total = len(EXPERIMENTS)
for i, e in enumerate(EXPERIMENTS, 1):
    tag = e['tag']
    dst_log = f'{RESULTS_DIR}/{e["scenario"]}/{tag}/training_log.csv'
    if SKIP_IF_EXISTS and os.path.isfile(dst_log):
        try:
            edf = pd.read_csv(dst_log)
            v_str = f'val_total={edf.val_total.min():.4f}'
        except Exception:
            v_str = 'unreadable'
        print(f'\n[{i}/{total}] [SKIP_EXIST] {tag}: {v_str}')
        run_results.append({'tag': tag, 'status':'skipped'})
        continue
    print(f'\n{"="*78}\n[{i}/{total}] {tag}: {e["desc"]} (scn={e["scenario"]} lr={e["lr"]})\n{"="*78}')
    t0 = time.time()
    r = subprocess.run(_build_cli(e), capture_output=False)
    el = time.time() - t0
    status = 'ok' if r.returncode == 0 else f'fail({r.returncode})'
    el_tot = time.time() - t_start
    done_now = sum(1 for x in run_results if x['status']!='skipped') + 1
    eta_min = (el_tot / max(done_now,1)) * (total - i) / 60
    print(f'\n[{i}/{total}] {tag} -> {status}  ({el/60:.1f}min)  ETA={eta_min:.0f}min')
    # PUSH per-run
    print(f'   pushing {tag}...')
    pushed = _push_run(e)
    run_results.append({'tag': tag, 'status': status, 'pushed': pushed, 'elapsed': el})

print(f'\n{"="*78}\nSWEEP DONE in {(time.time()-t_start)/60:.0f}min')
for r in run_results[-10:]:
    if r['status'] != 'skipped':
        print(f"   {r['tag']:<45} {r['status']:<10} push={r.get('pushed','N/A')}")


In [ ]:
# Cell 5 -- Aggregate analysis: leggi tutti i csv -> DataFrame unico
import pandas as pd, os, math, csv
RESULTS_DIR = 'results/Prodigy_Study/MultiParam'

rows = []
for e in EXPERIMENTS:
    tag = e['tag']
    base = f'{RESULTS_DIR}/{e["scenario"]}/{tag}'
    log = f'{base}/training_log.csv'
    blog = f'{base}/training_batch_log.csv'
    if not os.path.isfile(log):
        rows.append({'tag': tag, **{k: e[k] for k in ['scenario','lr','variant_id','d_coef','d0','growth_rate','scheduler','desc']},
                     'status':'MISSING'})
        continue
    df = pd.read_csv(log)
    bdf = pd.read_csv(blog) if os.path.isfile(blog) else None
    # Metrics
    vts = df['val_total'].values
    bi = int(df['val_total'].idxmin())
    # Grad explosion count (per-batch)
    if bdf is not None and 'gn_total_preclip' in bdf.columns:
        gn = bdf['gn_total_preclip']
        n_inf = int(gn.apply(lambda x: math.isinf(x) if isinstance(x,(int,float)) else False).sum())
        n_huge = int((gn > 1e6).sum())
        gn_explosion_pct = (n_inf + n_huge) / len(bdf) * 100
    else:
        gn_explosion_pct = float('nan')
    # lr_eff dynamics
    if bdf is not None and 'prodigy_lr_eff' in bdf.columns:
        lr_eff_start = float(bdf['prodigy_lr_eff'].iloc[0])
        lr_eff_max = float(bdf['prodigy_lr_eff'].max())
        lr_eff_end = float(bdf['prodigy_lr_eff'].iloc[-1])
    else:
        lr_eff_start=lr_eff_max=lr_eff_end=float('nan')
    row = {
        'tag': tag,
        'scenario': e['scenario'], 'lr': e['lr'],
        'variant_id': e['variant_id'], 'd_coef': e['d_coef'], 'd0': e['d0'],
        'growth_rate': e['growth_rate'], 'scheduler': e['scheduler'],
        'desc': e['desc'],
        'status':'OK', 'n_ep': len(df),
        'val_total_best': float(vts[bi]), 'best_ep': bi+1,
        'val_total_last': float(vts[-1]),
        'val_data_at_best': float(df['val_data'].iloc[bi]),
        'spike_rate_avg': float(df['spike_rate'].mean()),
        'gn_explosion_pct': gn_explosion_pct,
        'lr_eff_start': lr_eff_start, 'lr_eff_max': lr_eff_max, 'lr_eff_end': lr_eff_end,
    }
    rows.append(row)

df_all = pd.DataFrame(rows)
print(f'Total runs: {len(df_all)} (OK: {(df_all.status=="OK").sum()}, MISSING: {(df_all.status=="MISSING").sum()})')
os.makedirs(RESULTS_DIR, exist_ok=True)
df_all.to_csv(f'{RESULTS_DIR}/_aggregate.csv', index=False)
print(f'[saved] {RESULTS_DIR}/_aggregate.csv')
from IPython.display import display
display(df_all[df_all.status=='OK'].sort_values('val_total_best').head(15).round(4))


In [ ]:
# Cell 6 -- PARETO plots per ogni scenario (val_total vs grad_explosion_pct, val_total vs spike_rate, val_total vs lr_eff_max)
import matplotlib.pyplot as plt, os

ok = df_all[df_all.status=='OK'].copy()
os.makedirs('opt_plots/R24_pareto', exist_ok=True)

scen_colors = {'highway':'#1f77b4','mixed':'#ff7f0e','full':'#2ca02c'}

for scn in ['highway','mixed','full']:
    sub = ok[ok.scenario==scn].copy()
    if len(sub) == 0:
        print(f'[skip plot] {scn}: no OK runs')
        continue
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    # Panel 1: val_total vs grad_explosion_pct
    for lr_v in sub.lr.unique():
        ss = sub[sub.lr==lr_v]
        axes[0].scatter(ss.gn_explosion_pct, ss.val_total_best, s=80, alpha=0.7, label=f'lr={lr_v}')
        for _, r in ss.iterrows():
            axes[0].annotate(r.variant_id, (r.gn_explosion_pct, r.val_total_best), fontsize=7, alpha=0.7)
    axes[0].set_xlabel('gradient explosion %  (lower=better)')
    axes[0].set_ylabel('val_total best  (lower=better)')
    axes[0].set_title(f'{scn}: Pareto val_total vs grad_explosion')
    axes[0].grid(alpha=0.3); axes[0].legend()
    # Panel 2: val_total vs lr_eff_max
    for lr_v in sub.lr.unique():
        ss = sub[sub.lr==lr_v]
        axes[1].scatter(ss.lr_eff_max, ss.val_total_best, s=80, alpha=0.7, label=f'lr={lr_v}')
        for _, r in ss.iterrows():
            axes[1].annotate(r.variant_id, (r.lr_eff_max, r.val_total_best), fontsize=7, alpha=0.7)
    axes[1].set_xscale('log')
    axes[1].set_xlabel('lr_eff_max (log, target ~2e-3 stable)')
    axes[1].set_ylabel('val_total best')
    axes[1].set_title(f'{scn}: Pareto val_total vs lr_eff_max')
    axes[1].grid(alpha=0.3); axes[1].legend()
    # Panel 3: val_total vs spike_rate (target 15-20%)
    for lr_v in sub.lr.unique():
        ss = sub[sub.lr==lr_v]
        axes[2].scatter(ss.spike_rate_avg*100, ss.val_total_best, s=80, alpha=0.7, label=f'lr={lr_v}')
        for _, r in ss.iterrows():
            axes[2].annotate(r.variant_id, (r.spike_rate_avg*100, r.val_total_best), fontsize=7, alpha=0.7)
    axes[2].axvspan(15, 20, alpha=0.15, color='green', label='target FPGA 15-20%')
    axes[2].set_xlabel('spike rate avg %  (target 15-20%)')
    axes[2].set_ylabel('val_total best')
    axes[2].set_title(f'{scn}: Pareto val_total vs spike_rate')
    axes[2].grid(alpha=0.3); axes[2].legend()
    fig.suptitle(f'R2.4 Prodigy MultiParam -- scenario={scn} (30 run: 3 lr x 10 var)',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    out = f'opt_plots/R24_pareto/pareto_{scn}.png'
    fig.savefig(out, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'[saved] {out}')


In [ ]:
# Cell 7 -- Per scenario: TOP 5 best + WORST 5 + summary stats
from IPython.display import display, Markdown

for scn in ['highway','mixed','full']:
    sub = ok[ok.scenario==scn].sort_values('val_total_best')
    if len(sub) == 0: continue
    display(Markdown(f'## Scenario: {scn} ({len(sub)} run OK)'))
    cols_show = ['tag','lr','variant_id','d_coef','d0','growth_rate','scheduler',
                 'val_total_best','val_data_at_best','best_ep','spike_rate_avg',
                 'gn_explosion_pct','lr_eff_max']
    display(Markdown('### TOP 5 (best val_total)'))
    display(sub.head(5)[cols_show].round(4))
    display(Markdown('### WORST 5 (worst val_total)'))
    display(sub.tail(5)[cols_show].round(4))
    display(Markdown('### Stats per LR'))
    stats = sub.groupby('lr').agg(
        n=('val_total_best','count'),
        val_total_min=('val_total_best','min'),
        val_total_mean=('val_total_best','mean'),
        val_total_max=('val_total_best','max'),
        gn_expl_mean=('gn_explosion_pct','mean'),
        lr_eff_max_median=('lr_eff_max','median'),
    ).round(4)
    display(stats)
